<a href="https://colab.research.google.com/github/sotesh1516/Transformer-Attention_Is_All_You_Need/blob/main/Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import torch

The following are part of tokenization, converting text into discreet IDs.

In [ ]:
def build_token_to_id_vocab(sentences, specials=('<pad>', '<bos>', '<eos>', '<unk>')):
    # build a token-to-id dict with specials first, then corpus tokens in first-seen order.
    vocab_dict = {}
    global_id_counter = 0
    for sp in specials:
        vocab_dict[sp] = global_id_counter
        global_id_counter+=1
    for w in sentences:
        w_arr = w.split()
        for t in w_arr:
            if t not in vocab_dict:
                vocab_dict[t] = global_id_counter
                global_id_counter+=1

    return vocab_dict

def build_id_to_token_vocab(token_to_id):
    # build the inverse id-to-token dictionary from token_to_id
    id_to_token = {}
    for token in token_to_id:
        id_to_token[token_to_id[token]] = token

    return id_to_token

def encode_sentence_to_ids(sentence, token_to_id, unk_token='<unk>'):
    # convert whitespace tokens of `sentence` to ids via `token_to_id`, using `unk_token`'s id for OOV
    int_token_id = []
    whitespace_token = sentence.split()
    for t in whitespace_token:
        if t not in token_to_id:
            int_token_id.append(token_to_id[unk_token])
        else:
            int_token_id.append(token_to_id[t])

    return int_token_id

def decode_ids_to_tokens(ids, id_to_token):
    # map each id in ids to its token string via id_to_token and return the list
    tokens = []
    for id in ids:
        tokens.append(id_to_token[id])
    return tokens

def pad_id_sequence(ids, max_len, pad_id):
    # return a list of length exactly max_len, padding with pad_id or truncating.
    if max_len > len(ids):
        return ids + [pad_id for i in range(max_len - len(ids))]
    return ids[:max_len]

def stack_padded_sequences_to_batch(padded_sequences):
    """Stack a list of equal-length padded id sequences into a 2D LongTensor batch."""
    # stack padded id sequences into a (B, L) torch.long tensor
    return torch.tensor(padded_sequences).long()

Embeddings and Positional Encoding

In [ ]:
def scale_embeddings_by_sqrt_d_model(embeddings, d_model):
    """Scale a token embedding tensor by sqrt(d_model)."""
    # rescale embeddings by sqrt(d_model) as in the original Transformer paper
    return embeddings * math.sqrt(d_model)

def compute_positional_div_term(d_model):
    """
    Note:
    - Using several different frequencies gives each position a distinguishable vector pattern.
    - Assign each feature channel(dimension) its own angular frequency
    - Here we are producing pair-wise freq
        - frequency for pair 0 (dim 0 and 1)
        - frequency for pair 1
        - frequency for pair 2
    - Finally we can do pos * 1000^(-2i/d_model) -> this is the current step
    """
    # return a 1D FloatTensor of length d_model // 2 holding the sinusoidal frequency divisors
    output_len = d_model // 2
    i = torch.arange(output_len, dtype=torch.float32)
    freq = 10000 ** (-2.0 * i / d_model)
    return freq

def build_position_index_column(max_len):
    """
    Return a (max_len, 1) float tensor of [0, 1, ..., max_len-1].

    The sinusoidal positional encoding evaluates sine and cosine at every (position, frequency) pair.
    """
    # build a column vector of position indices from 0 to max_len-1
    pos_indices = torch.empty(max_len,1)
    for i in range(max_len):
        pos_indices[i] = torch.tensor(i).float()
    return pos_indices

def fill_even_indices_with_sin(pe, position, div_term):
    """Fill even feature indices of pe with sin(position * div_term)."""
    # write sin(position * div_term) into the even-indexed columns of pe and return it
    for i in range(len(pe)):
        for j in range(len(pe[0])):
            if j % 2 == 0: #even dimension
                current_dim_freq = j // 2
                pe[i, j] = torch.sin(position[i] * div_term[current_dim_freq])

    return pe

def fill_odd_indices_with_cos(pe, position, div_term):
    # fill the odd-indexed columns of pe with cos(position * div_term)
    for i in range(len(pe)):
        for j in range(len(pe[0])):
            if j % 2 != 0: #odd dimension
                current_dim_freq = j // 2
                pe[i, j] = torch.cos(position[i] * div_term[current_dim_freq])

    return pe

def build_sinusoidal_positional_encoding(max_len, d_model):
    """Assemble the (max_len, d_model) sinusoidal positional encoding matrix."""
    # build the (max_len, d_model) sinusoidal positional encoding matrix

    positional_encoding = torch.zeros(max_len, d_model)
    positional_index_cols = build_position_index_column(max_len) # max_len x 1
    divisor_terms = compute_positional_div_term(d_model)
    even_filled_positional_encoding = fill_even_indices_with_sin(positional_encoding, positional_index_cols, divisor_terms)
    positional_encoding = fill_odd_indices_with_cos(even_filled_positional_encoding, positional_index_cols, divisor_terms)
    return positional_encoding

def add_positional_encoding_to_embeddings(embedded_batch, positional_encoding):
    # add the first L rows of positional_encoding to embedded_batch and return the sum.
    # for each batch, position resets
    input_batch = embedded_batch.shape[0]
    num_input_token = embedded_batch.shape[1]
    d_model = embedded_batch.shape[2] #can get this from the pos encoding too
    positional_embedding = torch.empty(input_batch, num_input_token, d_model)
    for i in range(input_batch):
        positional_embedding[i] = embedded_batch[i] + positional_encoding[:num_input_token, :]

    return positional_embedding

Masks and Scaled Dot-Product Attention

In [ ]:
def build_padding_mask(token_ids, pad_id):
    """Return a (B, 1, 1, L) bool mask: True where token_ids != pad_id.
    - multiple batch
        - 1 Head
            - 1 query
                - multiple keys
    """
    # build a boolean mask marking non-pad positions, shaped for broadcasting against attention scores
    num_batches = token_ids.shape[0]
    num_keys = token_ids.shape[1]
    bool_mask = torch.empty((num_batches, 1, 1, num_keys), dtype=torch.bool)

    for b in range(num_batches):
        for key in range(num_keys):
            bool_mask[b, 0, 0, key] = token_ids[b, key] != pad_id

    return bool_mask

def build_causal_mask(seq_len):
    """Return a (1, 1, seq_len, seq_len) bool mask, True on and below diagonal."""
    # build a lower-triangular boolean causal mask of shape (1, 1, seq_len, seq_len)
    lower_triangular_bool_mask = torch.tril(
    torch.ones(seq_len, seq_len, dtype=torch.bool)
    )

    lower_triangular_bool_mask = lower_triangular_bool_mask.unsqueeze(0).unsqueeze(0)

    return lower_triangular_bool_mask

def combine_padding_and_causal_masks(padding_mask, causal_mask):
    # combine a (B,1,1,L) padding mask with a (1,1,L,L) causal mask into (B,1,L,L).
    # broadcasting works left to right, using a simple rule
      # same size OR one of them is 1
    return causal_mask & padding_mask

def compute_raw_attention_scores(query, key):
    """Compute raw attention scores Q @ K^T over the last two dimensions."""
    # matmul query with the transpose of key over the last two axes
    # 2 x 2 -> query
    # 3 x 2 -> key
    return query @ key.transpose(-2,-1)

def scale_attention_scores(scores, d_k):
    # divide raw attention scores by sqrt(d_k) to stabilize softmax inputs
    return scores / math.sqrt(d_k)

def mask_attention_scores_with_neg_inf(scores, mask):
    """Set entries of scores where mask is False to -inf."""
    # replace blocked positions of scores with negative infinity
    # masked_fill replaces positions where its mask is True, hence ~
    return scores.masked_fill(~mask, float("-inf"))

def softmax_attention_weights(masked_scores):
    # softmax over the last axis, zeroing rows that are entirely -inf
    init_prob =  torch.softmax(masked_scores, dim=-1)
    row_mask = torch.isneginf(masked_scores).all(dim=-1, keepdim=True)
    return torch.where(row_mask, 0.0, init_prob)

def apply_attention_weights_to_values(attention_weights, value):
    """Multiply attention weights by the value matrix to produce context vectors."""
    # combine attention weights (..., Lq, Lk) with value (..., Lk, d_v)
    return attention_weights @ value

def scaled_dot_product_attention(query, key, value, mask=None):
    """Run scaled dot-product attention; return (context, attention_weights)."""
    # chain raw scores, scale by sqrt(d_k), optionally mask, softmax, then mix values
    raw_weighted_score = compute_raw_attention_scores(query, key)
    scaled_attention_score = scale_attention_scores(raw_weighted_score, key.shape[-1])
    if mask != None:
        masked_attention_score = mask_attention_scores_with_neg_inf(scaled_attention_score, mask)
    else:
        masked_attention_score = scaled_attention_score

    softmax_attention_score = softmax_attention_weights(masked_attention_score)
    context_vector = apply_attention_weights_to_values(softmax_attention_score, value)
    return (context_vector, softmax_attention_score)

Multi-head Attention

In [ ]:
def split_last_dim_into_heads(tensor, num_heads):
    # reshape (B, L, d_model) into (B, L, num_heads, d_model // num_heads)
    batch, length, d_model = tensor.shape
    return tensor.reshape(batch, length, num_heads, d_model // num_heads)

def transpose_heads_before_sequence(split_tensor):
    # rearrange (B, L, num_heads, d_k) into (B, num_heads, L, d_k).
    return split_tensor.transpose(1,2)

def transpose_heads_before_sequence(split_tensor):
    # rearrange (B, L, num_heads, d_k) into (B, num_heads, L, d_k).
    # one attention-head at a time, instead of one token in each attention-head
    return split_tensor.transpose(1,2)

def merge_heads_back_to_model_dim(multi_head_tensor):
    # merge the head axis back into the feature axis to reconstruct d_model
    # switch num_heads and length first
    batch, num_heads, length, d_k = multi_head_tensor.shape
    return multi_head_tensor.transpose(1,2).reshape(batch, length, num_heads * d_k)

def apply_linear_projection(x, weight, bias):
    # return x @ weight^T + bias (bias may be None) with shape (..., out_features)
    # (batch, length, in_features) x (out_features, in_features).T
        # broadcasting is used for earlier dim (batch, length)
    return x @ weight.T + bias if bias is not None else x @ weight.T

def project_to_query_key_value(x, w_q, b_q, w_k, b_k, w_v, b_v):
    # project x into separate query, key, and value tensors via three linear layers
    query = apply_linear_projection(x, w_q, b_q)
    key = apply_linear_projection(x, w_k, b_k)
    value = apply_linear_projection(x, w_v, b_v)

    return query, key, value

def split_qkv_into_heads(q, k, v, num_heads):
    # split each of q, k, v into (B, num_heads, L, d_k) and return as a tuple
    q_reshaped_transposed = transpose_heads_before_sequence(split_last_dim_into_heads(q, num_heads))
    k_reshaped_transposed = transpose_heads_before_sequence(split_last_dim_into_heads(k, num_heads))
    v_reshaped_transposed = transpose_heads_before_sequence(split_last_dim_into_heads(v, num_heads))

    return q_reshaped_transposed, k_reshaped_transposed, v_reshaped_transposed

def multi_head_scaled_dot_product_attention(q_h, k_h, v_h, mask=None):
    # run scaled dot-product attention over per-head Q, K, V and return (context, weights)
    context_vector, softmax_attention_score = scaled_dot_product_attention(q_h, k_h, v_h, mask)
    return context_vector, softmax_attention_score

def merge_heads_and_project_output(context, w_o, b_o):
    # merge the head axis back into d_model and apply the output linear projection.
    single_head_context_tensor = merge_heads_back_to_model_dim(context)
    linear_output = apply_linear_projection(single_head_context_tensor, w_o, b_o)
    return linear_output

def assemble_multi_head_attention_forward(query, key, value, w_q, w_k, w_v, w_o, num_heads, mask=None):
    # project Q/K/V, split into heads, run scaled dot-product attention, merge heads, output projection.
    query_q, key_q, value_q = project_to_query_key_value(query, w_q, None, w_k, None, w_v, None)
    query_k, key_k, value_k = project_to_query_key_value(key, w_q, None, w_k, None, w_v, None)
    query_v, key_v, value_v = project_to_query_key_value(value, w_q, None, w_k, None, w_v, None)

    query, key, value = split_qkv_into_heads(query_q, key_k, value_v, num_heads)
    context_vector, softmax_attention_score = multi_head_scaled_dot_product_attention(query, key, value, mask)
    single_head_context_tensor = merge_heads_and_project_output(context_vector, w_o, None)

    return single_head_context_tensor